# Task 2: Painting Similarity Search
## GSoC 2025 — ArtExtract: Neural Networks for Artworks

This notebook implements a painting similarity search system using:
- **ResNet-50** pretrained backbone for feature extraction
- **Cosine similarity** over L2-normalized embeddings
- **National Gallery of Art Open Data** dataset
- **Precision@k, Recall@k, mAP** evaluation metrics

## 1. Setup & Dependencies

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

# Create results directory
import os
RESULTS_DIR = '/content/drive/MyDrive/Similarity_Results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Results will be saved to: {RESULTS_DIR}')

In [ ]:
!pip install -q matplotlib seaborn tqdm pandas Pillow

## 2. Download the NGA Open Data Dataset

We use the [National Gallery of Art Open Data](https://github.com/NationalGalleryOfArt/opendata) repository.
This contains metadata CSV files. For images, we'll download a sample from their IIIF API.

In [ ]:
# Clone the NGA Open Data repo for metadata
!git clone --depth 1 https://github.com/NationalGalleryOfArt/opendata.git /content/nga_opendata
!ls /content/nga_opendata/data/

In [ ]:
import pandas as pd

# Load the objects metadata
objects_df = pd.read_csv('/content/nga_opendata/data/objects.csv', low_memory=False)
print(f'Total objects in NGA: {len(objects_df)}')
print(f'Columns: {list(objects_df.columns)}')
objects_df.head()

In [ ]:
# Filter for paintings with IIIF image URLs
paintings = objects_df[
    (objects_df['classification'].str.contains('Painting', case=False, na=False)) &
    (objects_df['iiifthumburl'].notna())
].copy()

print(f'Paintings with images: {len(paintings)}')
print(f'\nTop artists:')
print(paintings['attribution'].value_counts().head(15))
print(f'\nClassifications:')
print(paintings['classification'].value_counts().head(10))

In [ ]:
import requests
from PIL import Image
from io import BytesIO
from tqdm import tqdm
import time

# Download painting images from IIIF API
# Limit to a manageable sample (e.g., 500 paintings) for GPU efficiency
MAX_IMAGES = 500
IMAGE_DIR = '/content/nga_images'
os.makedirs(IMAGE_DIR, exist_ok=True)

# Sample paintings, preferring those with known artists for evaluation
# Keep only artists with at least 3 paintings for meaningful evaluation
artist_counts = paintings['attribution'].value_counts()
valid_artists = artist_counts[artist_counts >= 3].index
eval_paintings = paintings[paintings['attribution'].isin(valid_artists)].copy()

# Sample up to MAX_IMAGES
if len(eval_paintings) > MAX_IMAGES:
    eval_paintings = eval_paintings.sample(n=MAX_IMAGES, random_state=42)

print(f'Selected {len(eval_paintings)} paintings from {eval_paintings["attribution"].nunique()} artists')

# Download images
downloaded = []
failed = 0

for idx, row in tqdm(eval_paintings.iterrows(), total=len(eval_paintings), desc='Downloading images'):
    thumb_url = row['iiifthumburl']
    obj_id = row['objectid']
    save_path = os.path.join(IMAGE_DIR, f'{obj_id}.jpg')
    
    # Skip if already downloaded
    if os.path.exists(save_path):
        downloaded.append({'objectid': obj_id, 'path': save_path, **row.to_dict()})
        continue
    
    try:
        # Modify IIIF URL to get a reasonable resolution (400px wide)
        iiif_url = thumb_url.replace('/full/!200,200/', '/full/!400,400/')
        resp = requests.get(iiif_url, timeout=10)
        if resp.status_code == 200:
            img = Image.open(BytesIO(resp.content)).convert('RGB')
            img.save(save_path, 'JPEG', quality=90)
            downloaded.append({'objectid': obj_id, 'path': save_path, **row.to_dict()})
        else:
            failed += 1
    except Exception as e:
        failed += 1
    
    # Be nice to the API
    time.sleep(0.1)

print(f'\nDownloaded: {len(downloaded)} | Failed: {failed}')

In [ ]:
# Save metadata for downloaded paintings
import json

metadata_path = os.path.join(IMAGE_DIR, 'metadata.json')
with open(metadata_path, 'w') as f:
    json.dump(downloaded, f, indent=2, default=str)
print(f'Saved metadata for {len(downloaded)} paintings')

## 3. Upload Similarity Code

Upload the Python modules from your local machine.

In [ ]:
# Option A: Upload files manually
# from google.colab import files
# uploaded = files.upload()  # Upload feature_extractor.py, similarity_search.py, evaluate_similarity.py

# Option B: Copy from Drive (if you've uploaded to Drive already)
# !cp /content/drive/MyDrive/Similarity/*.py /content/

# Option C: Write inline (the next cells contain the code directly)
print('Code modules are ready. Proceeding...')

## 4. Feature Extraction

Extract 2048-dimensional embeddings from all downloaded paintings using ResNet-50.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Feature extractor model
class PaintingFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])
        self.embedding_dim = 2048
    
    def forward(self, x):
        features = self.backbone(x).squeeze(-1).squeeze(-1)
        return torch.nn.functional.normalize(features, p=2, dim=1)

# Dataset
class PaintingDataset(Dataset):
    def __init__(self, image_paths, transform):
        self.image_paths = image_paths
        self.transform = transform
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        try:
            img = Image.open(self.image_paths[idx]).convert('RGB')
            img = self.transform(img)
        except:
            img = torch.zeros(3, 224, 224)
        return img, idx

# Setup
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Get all image paths
image_paths = [d['path'] for d in downloaded]

dataset = PaintingDataset(image_paths, transform)
dataloader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=2)

# Extract embeddings
model = PaintingFeatureExtractor().to(device)
model.eval()

all_embeddings = []
with torch.no_grad():
    for images, indices in tqdm(dataloader, desc='Extracting features'):
        images = images.to(device)
        embeddings = model(images)
        all_embeddings.append(embeddings.cpu().numpy())

embeddings = np.concatenate(all_embeddings, axis=0)
print(f'\nExtracted embeddings: {embeddings.shape}')  # (N, 2048)

# Save
np.savez_compressed(
    os.path.join(RESULTS_DIR, 'embeddings.npz'),
    embeddings=embeddings,
    image_paths=np.array(image_paths),
    filenames=np.array([os.path.basename(p) for p in image_paths])
)
print('Embeddings saved!')

## 5. Similarity Search

Find the most similar paintings for random queries using cosine similarity.

In [ ]:
import matplotlib.pyplot as plt

def find_similar(query_idx, embeddings, k=5):
    """Find k most similar paintings using cosine similarity."""
    similarities = embeddings @ embeddings[query_idx]
    top_indices = np.argsort(similarities)[::-1][1:k+1]  # Exclude self
    return top_indices, similarities[top_indices]

def show_similar(query_idx, embeddings, image_paths, metadata, k=5):
    """Visualize query and its most similar paintings."""
    top_indices, similarities = find_similar(query_idx, embeddings, k)
    
    fig, axes = plt.subplots(1, k+1, figsize=(4*(k+1), 5))
    
    # Query
    img = Image.open(image_paths[query_idx]).convert('RGB')
    axes[0].imshow(img)
    q_meta = metadata[query_idx]
    title = q_meta.get('title', 'Unknown')[:30]
    artist = str(q_meta.get('attribution', 'Unknown'))[:25]
    axes[0].set_title(f'QUERY\n{title}\n{artist}', fontsize=8, fontweight='bold', color='red')
    axes[0].axis('off')
    
    # Similar paintings
    for i, (idx, sim) in enumerate(zip(top_indices, similarities)):
        img = Image.open(image_paths[idx]).convert('RGB')
        axes[i+1].imshow(img)
        m = metadata[idx]
        t = str(m.get('title', ''))[:30]
        a = str(m.get('attribution', ''))[:25]
        axes[i+1].set_title(f'#{i+1} (sim: {sim:.3f})\n{t}\n{a}', fontsize=7)
        axes[i+1].axis('off')
    
    plt.suptitle('Painting Similarity Search', fontsize=14, fontweight='bold')
    plt.tight_layout()
    return fig

# Run similarity search for random queries
np.random.seed(42)
num_demos = 10
query_indices = np.random.choice(len(embeddings), num_demos, replace=False)

for i, q_idx in enumerate(query_indices):
    fig = show_similar(q_idx, embeddings, image_paths, downloaded, k=5)
    fig.savefig(os.path.join(RESULTS_DIR, f'similar_{i+1}.png'), dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()

print(f'\nSaved {num_demos} similarity visualizations to {RESULTS_DIR}')

## 6. Evaluation Metrics

Evaluate similarity search using artist labels as ground-truth relevance.
Two paintings by the same artist are considered "relevant" to each other.

In [ ]:
from collections import defaultdict

def precision_at_k(relevant, retrieved, k):
    return sum(1 for idx in retrieved[:k] if idx in relevant) / k

def recall_at_k(relevant, retrieved, k):
    if not relevant:
        return 0.0
    return sum(1 for idx in retrieved[:k] if idx in relevant) / len(relevant)

def average_precision(relevant, retrieved, max_k=50):
    retrieved = retrieved[:max_k]
    if not relevant:
        return 0.0
    precisions = []
    hits = 0
    for i, idx in enumerate(retrieved):
        if idx in relevant:
            hits += 1
            precisions.append(hits / (i + 1))
    return sum(precisions) / len(relevant) if precisions else 0.0

# Build artist groups for evaluation
labels = [str(d.get('attribution', 'unknown')) for d in downloaded]
artist_groups = defaultdict(set)
for i, label in enumerate(labels):
    if label != 'unknown' and label != 'nan':
        artist_groups[label].add(i)

# Only evaluate queries that have at least 1 other painting by the same artist
valid_queries = [i for i in range(len(labels)) 
                 if labels[i] in artist_groups and len(artist_groups[labels[i]]) > 1]

print(f'Valid queries (artists with 2+ paintings): {len(valid_queries)}')
print(f'Artist groups: {len(artist_groups)}')

# Evaluate
k_values = [1, 3, 5, 10, 20]
prec = {k: [] for k in k_values}
rec = {k: [] for k in k_values}
ap_scores = []

for q_idx in tqdm(valid_queries, desc='Evaluating'):
    relevant = artist_groups[labels[q_idx]] - {q_idx}  # Same artist, excluding self
    top_indices, _ = find_similar(q_idx, embeddings, k=max(k_values))
    retrieved = top_indices.tolist()
    
    for k in k_values:
        prec[k].append(precision_at_k(relevant, retrieved, k))
        rec[k].append(recall_at_k(relevant, retrieved, k))
    ap_scores.append(average_precision(relevant, retrieved))

# Print results
mAP = np.mean(ap_scores)
print(f'\n{"="*60}')
print(f'  SIMILARITY EVALUATION RESULTS')
print(f'{"="*60}')
print(f'  Queries: {len(valid_queries)}')
print(f'  Mean Average Precision (mAP): {mAP*100:.2f}%')
print()
print(f'{"k":>6s} | {"Prec@k":>10s} | {"Rec@k":>10s}')
print(f'{"-"*6}-+-{"-"*10}-+-{"-"*10}')
for k in k_values:
    p = np.mean(prec[k]) * 100
    r = np.mean(rec[k]) * 100
    print(f'{k:6d} | {p:9.2f}% | {r:9.2f}%')

In [ ]:
# Plot evaluation metrics
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

precisions = [np.mean(prec[k]) * 100 for k in k_values]
recalls = [np.mean(rec[k]) * 100 for k in k_values]

ax1.bar(range(len(k_values)), precisions, color='steelblue', alpha=0.85)
ax1.set_xticks(range(len(k_values)))
ax1.set_xticklabels([f'P@{k}' for k in k_values])
ax1.set_ylabel('Precision (%)')
ax1.set_title('Precision@k')
ax1.set_ylim(0, 100)
for i, v in enumerate(precisions):
    ax1.text(i, v + 1.5, f'{v:.1f}', ha='center', fontsize=10)

ax2.bar(range(len(k_values)), recalls, color='coral', alpha=0.85)
ax2.set_xticks(range(len(k_values)))
ax2.set_xticklabels([f'R@{k}' for k in k_values])
ax2.set_ylabel('Recall (%)')
ax2.set_title('Recall@k')
ax2.set_ylim(0, 100)
for i, v in enumerate(recalls):
    ax2.text(i, v + 1.5, f'{v:.1f}', ha='center', fontsize=10)

plt.suptitle(f'Similarity Retrieval Evaluation (mAP: {mAP*100:.1f}%)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'evaluation_metrics.png'), dpi=150)
plt.show()
print('Evaluation plot saved!')

In [ ]:
# Save all metrics as JSON
metrics = {
    'num_queries': len(valid_queries),
    'num_images': len(embeddings),
    'num_artist_groups': len(artist_groups),
    'embedding_dim': int(embeddings.shape[1]),
    'mAP': float(mAP),
}
for k in k_values:
    metrics[f'precision@{k}'] = float(np.mean(prec[k]))
    metrics[f'recall@{k}'] = float(np.mean(rec[k]))

with open(os.path.join(RESULTS_DIR, 'similarity_metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)

print('\nFinal metrics saved to Google Drive!')
print(json.dumps(metrics, indent=2))

## 7. Summary

### Architecture
- **Feature Extractor**: ResNet-50 (ImageNet-pretrained) → 2048-dim L2-normalized embeddings
- **Similarity Metric**: Cosine similarity (dot product on normalized vectors)
- **Ground Truth**: Paintings by the same artist are considered "relevant"

### Key Results
- Mean Average Precision (mAP) and Precision/Recall@k are reported above
- Visual examples demonstrate meaningful similarity retrieval
- The system finds paintings with similar visual characteristics (composition, palette, subject matter)